# 5 · NDWI — from wet/dry pixels to elevation logic

### The water index that powers intertidal height mapping

Page 3 tagged each Landsat pass with a tide height. This page picks two **clear-sky**
scenes at opposite tides and makes the wet/dry contrast **quantitative** using **NDWI** (Normalised
Difference Water Index).

**What this notebook answers**

1. What is NDWI, and why does it separate water from sediment?
2. How does NDWI **change** between a clear low-tide and high-tide Landsat scene?
3. How does that logic scale to an **elevation map** on page 6?

**The core idea (one sentence)**

> For each pixel, find the tide height where the surface switches from dry to wet —
> that crossover height **is** the sediment elevation.

Page 6 runs the full DEA algorithm (`intertidal.elevation()`). Here we build the
**intuition** with two clear-sky scenes so that output makes sense.

**Previous:** [4 · Validation](04_validation.ipynb) (optional) · **Next:** [6 · Elevation](06_elevation.ipynb)


## Step 1 — Imports

Same Planetary Computer stack as pages 2–4, plus **`cache_utils`** for the scene
catalogue and tide tags from page 3.


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pystac_client
import planetary_computer as pc
import odc.stac

from cache_utils import load_or_compute_scenes, fetch_stac_items_for_scenes

warnings.filterwarnings("ignore")
print("Imports OK.")


## Step 2 — Configuration

Same site as pages 3–4. We use **Landsat 8/9** only here — the same sensor family as
the elevation notebook (page 6).

This page prioritises **clear-sky scenes** so the RGB and NDWI maps are easy to read
(tight `MAX_CLOUD` keeps maps readable).

| Variable | Role |
|---|---|
| `MAX_CLOUD` | Scene-level cloud ceiling (%) — only scenes below this are used |
| `LOW_PCT` / `HIGH_PCT` | Pick extreme low / high tide scenes (same idea as page 3 tide percentiles) |
| `NDWI_WET` | Reference threshold (water > 0 is a common rule of thumb) |


In [ ]:
# === EDIT THESE ===
LON        = 4.81050
LAT        = 52.98886
SITE_NAME  = "WaddenSea"

DELTA      = 0.08
START      = "2023-01-01"
END        = "2023-12-31"

TIDE_DIR   = "./tide_models"
TIDE_MODEL = "FES2022"

LOW_PCT    = 15
HIGH_PCT   = 15
MAX_CLOUD  = 15        # clear skies only (keep maps readable)
NDWI_WET   = 0.0         # values above → classified as "wet" in Step 6 overlay

OVERWRITE_SCENES = False

# --- derived ---
os.environ["EO_TIDES_TIDE_MODELS"] = TIDE_DIR
BBOX = (LON - DELTA, LAT - DELTA, LON + DELTA, LAT + DELTA)
PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

print(f"Site   : {SITE_NAME}  ({LAT:.4f} N, {LON:.4f} E)")
print(f"Window : {START} to {END}")
print(f"Cloud  : keep scenes with eo:cloud_cover < {MAX_CLOUD}%")


## Step 3 — Select clear low- and high-tide scenes

We reuse the scene cache from page 3 (Landsat 8/9), then keep only **low-cloud**
passes (`MAX_CLOUD`).

For the **map comparison** (Steps 5–6) we pick the **clearest** scene in each tidal
class (low vs high tide).


In [ ]:
scenes_df = load_or_compute_scenes(
    lon=LON, lat=LAT,
    start=START, end=END,
    site_name=SITE_NAME,
    tide_model=TIDE_MODEL,
    tide_dir=TIDE_DIR,
    max_cloud=MAX_CLOUD,
    delta=DELTA,
    overwrite=OVERWRITE_SCENES,
    sensors=["Landsat 8", "Landsat 9"],
)

if "time" not in scenes_df.columns:
    scenes_df = scenes_df.reset_index()

usable = scenes_df[
    scenes_df["sensor"].isin(["Landsat 8", "Landsat 9"]) &
    (scenes_df["cloud_cover"].fillna(100) < MAX_CLOUD) &
    scenes_df["tide_height"].notna()
].copy()

if len(usable) < 2:
    raise SystemExit(
        f"Need ≥2 Landsat scenes with cloud < {MAX_CLOUD}% — "
        "extend END, raise MAX_CLOUD slightly, or set OVERWRITE_SCENES=True."
    )


def pick_clearest(candidates: pd.DataFrame, tide_target: float) -> pd.DataFrame:
    """Lowest cloud cover; tie-break by closest tide height to target."""
    c = candidates.copy()
    c["_cloud"] = c["cloud_cover"].fillna(100)
    c["_tdist"] = (c["tide_height"] - tide_target).abs()
    return c.sort_values(["_cloud", "_tdist"]).iloc[:1].drop(columns=["_cloud", "_tdist"])


low_thresh  = usable["tide_height"].quantile(LOW_PCT / 100)
high_thresh = usable["tide_height"].quantile(1 - HIGH_PCT / 100)

low_scenes  = usable[usable["tide_height"] <= low_thresh]
high_scenes = usable[usable["tide_height"] >= high_thresh]

if low_scenes.empty or high_scenes.empty:
    raise SystemExit("Need both low- and high-tide scenes — adjust LOW_PCT / HIGH_PCT or extend END.")

low_pick  = pick_clearest(low_scenes, low_thresh)
high_pick = pick_clearest(high_scenes, high_thresh)

print(f"Landsat scenes (cloud < {MAX_CLOUD}%): {len(usable)}")
print(
    f"Low-tide pick  : {low_pick['time'].iloc[0]}  "
    f"({low_pick['tide_height'].iloc[0]:+.2f} m, cloud {low_pick['cloud_cover'].iloc[0]:.0f}%)"
)
print(
    f"High-tide pick : {high_pick['time'].iloc[0]}  "
    f"({high_pick['tide_height'].iloc[0]:+.2f} m, cloud {high_pick['cloud_cover'].iloc[0]:.0f}%)"
)


## Step 4 — What is NDWI?

**NDWI** (Normalised Difference Water Index) compares **green** and **near-infrared**
reflectance:

$$
\mathrm{NDWI} = \frac{\mathrm{Green} - \mathrm{NIR}}{\mathrm{Green} + \mathrm{NIR}}
$$

| Surface | Green | NIR | NDWI |
|---|---|---|---|
| **Open water** | moderate | very low ( absorbed ) | **positive** (wet) |
| **Dry sediment** | moderate | higher | **negative** (dry) |
| **Vegetated marsh** | moderate | high | negative |

On an **intertidal flat**, the *same pixel* moves along this scale as the tide rises:
dry at low water → wet at high water.

**Why not RGB alone?** True-colour images are intuitive but subjective. NDWI gives a
**single number per pixel** that `intertidal.elevation()` can relate to **modelled tide
height** — that is the bridge to metres above sea level.

> DEA uses the continuous NDWI signal, not a hard wet/dry mask. The threshold
> `NDWI_WET = 0` in Step 6 is only for visual guidance.


## Step 5 — Load Landsat and compute NDWI

Helper below: STAC → surface reflectance → cloud mask → NDWI.

Landsat Collection 2 bands: `green`, `nir08`, `qa_pixel` (cloud/shadow bits 3+4).


In [ ]:
catalog = pystac_client.Client.open(PC_URL, modifier=pc.sign_inplace)


def load_landsat_ndwi(scenes: pd.DataFrame, label: str) -> xr.Dataset:
    """Load Landsat scenes and return reflectance + NDWI (+ RGB for display)."""
    items = fetch_stac_items_for_scenes(catalog, scenes, BBOX)
    if not items:
        raise ValueError(f"No STAC items for {label}")

    print(f"  {label}: {len(items)} scene(s) ...")
    ds = odc.stac.load(
        items,
        bands=["red", "green", "blue", "nir08", "qa_pixel"],
        bbox=BBOX,
        crs="utm",
        resolution=30,
        groupby="time",
        chunks={"x": 1024, "y": 1024},
        resampling={"qa_pixel": "nearest", "*": "cubic"},
        patch_url=pc.sign,
    ).compute()

    # Cloud + shadow mask (Landsat C2 QA)
    qa = ds["qa_pixel"]
    clear = np.bitwise_and(qa, (1 << 3) | (1 << 4)) == 0
    ds = ds.where(clear).drop_vars("qa_pixel")

    # USGS surface-reflectance scale
    for band in ("red", "green", "blue", "nir08"):
        ds[band] = (ds[band] * 0.0000275 - 0.2).clip(0, 1)

    ds["ndwi"] = (ds.green - ds.nir08) / (ds.green + ds.nir08)
    return ds


print("Loading low-tide scene ...")
ds_low  = load_landsat_ndwi(low_pick,  "low tide")
print("Loading high-tide scene ...")
ds_high = load_landsat_ndwi(high_pick, "high tide")
print("Done.")


## Step 6 — Compare NDWI at low vs high tide

Three columns per row:

1. **RGB** — true-colour view at each tide
2. **NDWI map** — blue = wet, brown = dry (RdBu diverging)
3. **Wet mask** — pixels with NDWI > `NDWI_WET`

Look for intertidal areas that flip from dry (low tide) to wet (high tide).


In [ ]:
def _rgb(da_red, da_green, da_blue, pct=(2, 98)):
    # Percentile stretch for display
    stacks = xr.concat([da_red, da_green, da_blue], dim="band")
    lo, hi = np.nanpercentile(stacks.values, pct)
    rgb = np.clip(stacks.values, lo, hi)
    rgb = (rgb - lo) / (hi - lo + 1e-9)
    return np.moveaxis(rgb, 0, -1)


def plot_ndwi_pair(ds_lo, ds_hi, tide_lo, tide_hi):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))

    for row, (ds, tide, title) in enumerate([
        (ds_lo, tide_lo, "Low tide"),
        (ds_hi, tide_hi, "High tide"),
    ]):
        t = 0 if ds.sizes.get("time", 1) == 1 else slice(None)
        red, green, blue = ds.red.isel(time=t), ds.green.isel(time=t), ds.blue.isel(time=t)
        ndwi = ds.ndwi.isel(time=t)
        wet = ndwi > NDWI_WET

        # RGB
        rgb = _rgb(red, green, blue)
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"{title}\nRGB ({tide:+.2f} m MSL)")

        # NDWI
        im = axes[row, 1].imshow(ndwi.values, cmap="RdBu", vmin=-0.5, vmax=0.5)
        axes[row, 1].set_title("NDWI")
        plt.colorbar(im, ax=axes[row, 1], fraction=0.046, label="NDWI")

        # Wet mask
        axes[row, 2].imshow(wet.values, cmap="Blues", vmin=0, vmax=1)
        axes[row, 2].set_title(f"NDWI > {NDWI_WET} (wet)")

        for ax in axes[row]:
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(f"{SITE_NAME} — NDWI at opposite tides", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(f"{SITE_NAME}_ndwi_low_high.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {SITE_NAME}_ndwi_low_high.png")


plot_ndwi_pair(
    ds_low, ds_high,
    float(low_pick["tide_height"].iloc[0]),
    float(high_pick["tide_height"].iloc[0]),
)


### Interpreting Step 6

| What you see | Meaning |
|---|---|
| **Intertidal flat dry at low tide, wet at high** | NDWI working as expected — these pixels carry elevation information |
| **Channel always wet** | Permanent water — no crossover → no elevation (subtidal) |
| **Land always dry** | Never floods → outside intertidal zone |
| **RGB and NDWI agree** | Wet/dry pattern visible in both columns |

If low and high NDWI maps look **identical**, check tide tags (page 3/4) or pick a wider percentile window.

If maps still look **hazy or patchy**, tighten `MAX_CLOUD` (e.g. 10) or extend `END` so Step 3 can find clearer passes.


## Step 7 — From NDWI stack to elevation map

For **one intertidal pixel**, imagine plotting NDWI (y-axis) against tide height
(x-axis) for every clear Landsat overpass:

```
NDWI
  +1 │     ●  ●      ← wet observations (high tide)
     │   ●
   0 │- - - - - - - -  ← crossover (waterline)
     │ ●
  -1 │●  ●            ← dry observations (low tide)
     └────────────────── tide height (m, MSL)
                      ↑
                 elevation estimate
```

`intertidal.elevation()` (page 6) does this statistically for **every pixel**:

1. Build a multi-year **NDWI time series** (many dots in the sketch)
2. Tag each observation with **FES2022** tide height
3. Fit the **crossover** + uncertainty
4. Export **elevation** (m, MSL) and **QA layers**

You already validated tide tags (pages 3–4) and saw wet/dry contrast in RGB and NDWI on this page. Page 6 is the full production run.


## Step 8 — What you have seen (recap)

1. **NDWI formula (Step 4)** — green vs NIR separates water from dry sediment
2. **Low vs high maps (Step 6)** — the same flat flips from negative to positive NDWI
3. **Crossover logic (Step 7)** — many (NDWI, tide) pairs per pixel → elevation on page 6

### Output figure

| File | Content |
|---|---|
| `SITE_NAME_ndwi_low_high.png` | RGB + NDWI + wet mask at low & high tide |

### Before page 6

| Check | Where |
|---|---|
| Enough Landsat scenes over several years | Page 3 |
| FES2022 trustworthy | Page 4 (optional) |
| Visible wet/dry contrast | This page (Step 6 maps) |

**Next:** [6 · Elevation](06_elevation.ipynb) — full intertidal DEM with `intertidal.elevation()`.
